In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import tqdm
import functools
import pandas as pd
from functools import partial
import time
from scipy.integrate import odeint
import torch.utils.data as data
import copy
import math

# Lorenz96 system definition
def lorenz96(x, t, F = 8.0):
    """Lorenz 96 model with constant forcing"""
    return (np.roll(x, -1) - np.roll(x, 2)) * np.roll(x, 1) - x + F 


def initial_state_gen(N, lower_bound = 0, upper_bound = 1):
    if upper_bound <= lower_bound:
        return np.ones(N)
    return np.random.uniform(lower_bound, upper_bound, N)

def lorenz96_solver(initial_state, terminal_time, time_step, F = 8.0):
    t = np.arange(0.0, terminal_time, time_step)
    #sol = solve_ivp(lorenz96, [t[0], t[-1]], initial_state, t_eval = t)
    #return sol
    sol = odeint(lorenz96, initial_state, t)
    return t, sol

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(data.shape[0] - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+seq_len, :])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

def create_data(data, LSTM_seq_len, device):
    data_input, data_output = create_sequences(data, LSTM_seq_len)
    data_input = data_input.to(device)
    #data_input = data_input.transpose(-2,-1).to(device)
    data_output = data_output.to(device)
    return data_input, data_output

class LSTMModel(nn.Module):
    def __init__(self, input_dim=3, output_dim=3, hidden_dim=128, num_layers=5):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden)

def training_LSTM(model, epochs, optimizer, loss_fn, data_input, data_output, batch_size = 128):
    loader = data.DataLoader(data.TensorDataset(data_input, data_output), shuffle=True, batch_size = batch_size)
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in loader:  
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 250 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")


def training_LSTM_main(initial_state, x_dim, time_step, terminal_time, LSTM_seq_len, model, epochs, optimizer, loss_fn, batch_size = 128, F = 8.0, device = 'cpu'):
    t, sol = lorenz96_solver(initial_state, terminal_time, time_step, F)
    data_input, data_output = create_data(sol, LSTM_seq_len, device)
    #current = time.time()
    training_LSTM(model, epochs, optimizer, loss_fn, data_input, data_output, batch_size)
    #print(f"Training time: {time.time() - current}s")

def training_LSTM_inaccurate(initial_state, x_dim, time_step, terminal_time, LSTM_seq_len, model, epochs, optimizer, loss_fn, noise_std = 1, batch_size = 128, F = 8.0, device = 'cpu'):
    t, sol = lorenz96_solver(initial_state, terminal_time, time_step, F)
    data_input, data_output = create_data(sol, LSTM_seq_len, device)
    data_output = data_output + torch.randn_like(data_output) * noise_std
    training_LSTM(model, epochs, optimizer, loss_fn, data_input, data_output, batch_size)

def LSTM_plotting(initial_state, x_dim, time_step, terminal_time, LSTM_seq_len, model, F = 8.0, device = 'cpu'):
    t, sol = lorenz96_solver(initial_state, terminal_time, time_step, F)
    data_input, data_output = create_data(sol, LSTM_seq_len, device)
    model.eval()
    with torch.no_grad():
        predictions = model(data_input).cpu().numpy()

    for i in range(dim_x):
        if_plot = np.random.randint(2)
        if if_plot:
            plt.plot(t[LSTM_seq_len: ], sol[LSTM_seq_len: , i], label = 'dim ' + str(i + 1), color = 'blue')
            plt.plot(t[LSTM_seq_len: ], predictions[:, i],  label = 'prediction', color = 'red')
            plt.legend(loc = 'upper right')
            plt.show()

def testing_dataset_MSE(initial_state, x_dim, time_step, terminal_time, LSTM_seq_len, model, F = 8.0, device = 'cpu'):
    t, sol = lorenz96_solver(initial_state, terminal_time, time_step, F)
    data_input, data_output = create_data(sol, LSTM_seq_len, device)
    model.eval()
    with torch.no_grad():
        predictions = model(data_input).cpu()

    return loss_fn(predictions, torch.Tensor(sol[LSTM_seq_len:, :]))

In [2]:
dim_x = 20
F = 8.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LSTMModel(input_dim = dim_x, output_dim = dim_x).to(device)
loss_fn = nn.MSELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
initial_state = np.ones(dim_x) * 8.0
initial_state[0] += 0.5
total_data_size = 2500
terminal_time = 10
time_step = terminal_time / total_data_size
LSTM_seq_len = 20
EPOCHS = 200
batch_size = 128
np_seed = 31415
device

'cuda'

In [3]:
train_test_loss = []
train_test_initial = initial_state_gen(dim_x, 7, 9)
train_test_loss_fix = []
initial_state_training = []

In [4]:
np.random.seed(0)
np.random.uniform(7, 9, dim_x)

array([8.09762701, 8.43037873, 8.20552675, 8.08976637, 7.8473096 ,
       8.29178823, 7.87517442, 8.783546  , 8.92732552, 7.76688304,
       8.58345008, 8.05778984, 8.13608912, 8.85119328, 7.14207212,
       7.1742586 , 7.04043679, 8.66523969, 8.5563135 , 8.7400243 ])

In [5]:
current_time = time.time()
noise_std = 0.2
for i in range(500):
    if (i + 1) % 20 == 0:
        print("Iteration", i + 1)
    if (i + 1) % 100 == 0:
        print("Time:", (time.time() - current_time) / 60)
        current_time = time.time()
    np.random.seed(np_seed + i)
    #initial_state = initial_state_gen(dim_x, 7, 9)
    initial_state = np.random.uniform(7, 9, dim_x)
    initial_state_training.append(initial_state)
    training_LSTM_main(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, EPOCHS, optimizer, loss_fn, batch_size, F, device)
    train_test_loss.append(testing_dataset_MSE(initial_state_gen(dim_x, 7, 9), dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    train_test_loss_fix.append(testing_dataset_MSE(train_test_initial, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    if (i + 1) % 100 == 0:
        torch.save(model.state_dict(), 'LSTM_Lorenz96_EnSF_accurate_' + str((i + 1) // 100) + '00.pth')
#LSTM_plotting(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device)

Iteration 20
Iteration 40
Iteration 60
Iteration 80
Iteration 100
Time: 42.97937047481537
Iteration 120
Iteration 140
Iteration 160
Iteration 180
Iteration 200
Time: 43.30079064369202
Iteration 220


KeyboardInterrupt: 

In [26]:
df = pd.DataFrame({'LSTM_training_loss_random_initial': train_test_loss, 
                   'LSTM_training_loss_fixed_initial': train_test_loss_fix})

df.to_csv('LSTM_training_loss_accurate_500.csv', index = False)
df.head()

,LSTM_training_loss_random_initial,LSTM_training_loss_fixed_initial
0,20.371775,14.889625
1,11.936386,15.510914
2,16.574009,14.515489
3,14.448654,12.298017
4,13.471327,12.357531


In [27]:
initial_state_training = np.array(initial_state_training)
np.savetxt('initial_state_500.txt', initial_state_training)

In [28]:
train_test_loss = []
train_test_initial = initial_state_gen(dim_x, 7, 9)
train_test_loss_fix = []
initial_state_training = []

In [29]:
current_time = time.time()
noise_std = 0.2
for i in range(500, 1000):
    if (i + 1) % 20 == 0:
        print("Iteration", i + 1)
    if (i + 1) % 100 == 0:
        print("Time:", (time.time() - current_time) / 60)
        current_time = time.time()
    np.random.seed(np_seed + i)
    #initial_state = initial_state_gen(dim_x, 7, 9)
    initial_state = np.random.uniform(7, 9, dim_x)
    initial_state_training.append(initial_state)
    training_LSTM_main(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, EPOCHS, optimizer, loss_fn, batch_size, F, device)
    train_test_loss.append(testing_dataset_MSE(initial_state_gen(dim_x, 7, 9), dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    train_test_loss_fix.append(testing_dataset_MSE(train_test_initial, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    if (i + 1) % 100 == 0:
        torch.save(model.state_dict(), 'LSTM_Lorenz96_EnSF_accurate_' + str((i + 1) // 100) + '00.pth')
#LSTM_plotting(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device)

Iteration 520
Iteration 540
Iteration 560
Iteration 580
Iteration 600
Time: 20.751920731862388
Iteration 620
Iteration 640
Iteration 660
Iteration 680
Iteration 700
Time: 20.82370700041453
Iteration 720
Iteration 740
Iteration 760
Iteration 780
Iteration 800
Time: 20.80733319123586
Iteration 820
Iteration 840
Iteration 860
Iteration 880
Iteration 900
Time: 20.771511165301003
Iteration 920
Iteration 940
Iteration 960
Iteration 980
Iteration 1000
Time: 20.69222801923752


In [30]:
df = pd.DataFrame({'LSTM_training_loss_random_initial_1000': train_test_loss, 
                   'LSTM_training_loss_fixed_initial_1000': train_test_loss_fix})

df.to_csv('LSTM_training_loss_accurate_1000.csv', index = False)
df.head()

,LSTM_training_loss_random_initial_1000,LSTM_training_loss_fixed_initial_1000
0,0.16537432,0.27617976
1,0.28962982,0.28318933
2,0.19972947,0.25916418
3,0.26186696,0.30714887
4,0.23782212,0.297456


In [31]:
initial_state_training = np.array(initial_state_training)
np.savetxt('initial_state_1000.txt', initial_state_training)

In [7]:
model.load_state_dict(torch.load('LSTM_Lorenz96_EnSF_accurate_1000.pth'))

<All keys matched successfully>

In [8]:
train_test_loss = []
train_test_initial = initial_state_gen(dim_x, 7, 9)
train_test_loss_fix = []
initial_state_training = []

In [9]:
current_time = time.time()
noise_std = 0.2
for i in range(1000, 1500):
    if (i + 1) % 20 == 0:
        print("Iteration", i + 1)
    if (i + 1) % 100 == 0:
        print("Time:", (time.time() - current_time) / 60)
        current_time = time.time()
    np.random.seed(np_seed + i)
    #initial_state = initial_state_gen(dim_x, 7, 9)
    initial_state = np.random.uniform(7, 9, dim_x)
    initial_state_training.append(initial_state)
    training_LSTM_main(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, EPOCHS, optimizer, loss_fn, batch_size, F, device)
    train_test_loss.append(testing_dataset_MSE(initial_state_gen(dim_x, 7, 9), dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    train_test_loss_fix.append(testing_dataset_MSE(train_test_initial, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device).detach().numpy())
    if (i + 1) % 100 == 0:
        torch.save(model.state_dict(), 'LSTM_Lorenz96_EnSF_accurate_' + str((i + 1) // 100) + '00.pth')
#LSTM_plotting(initial_state, dim_x, time_step, terminal_time, LSTM_seq_len, model, F, device)

Iteration 1020
Iteration 1040
Iteration 1060
Iteration 1080
Iteration 1100
Time: 40.59725952943166
Iteration 1120
Iteration 1140
Iteration 1160
Iteration 1180
Iteration 1200
Time: 43.48504234155019
Iteration 1220
Iteration 1240
Iteration 1260
Iteration 1280
Iteration 1300
Time: 43.73210206429164
Iteration 1320
Iteration 1340
Iteration 1360
Iteration 1380
Iteration 1400
Time: 43.644315715630846
Iteration 1420
Iteration 1440
Iteration 1460
Iteration 1480
Iteration 1500
Time: 43.172500709692635


In [10]:
df = pd.DataFrame({'LSTM_training_loss_random_initial_1500': train_test_loss, 
                   'LSTM_training_loss_fixed_initial_1500': train_test_loss_fix})

df.to_csv('LSTM_training_loss_accurate_1500.csv', index = False)
df.head()

,LSTM_training_loss_random_initial_1500,LSTM_training_loss_fixed_initial_1500
0,0.09938516,0.12227074
1,0.072403945,0.13366722
2,0.14810388,0.14055464
3,0.16597132,0.13200681
4,0.10086799,0.14365034


In [11]:
initial_state_training = np.array(initial_state_training)
np.savetxt('initial_state_1500.txt', initial_state_training)